# Welcome to NER Finetuning with HF Transformers 

First of all, we need to import all necessary libraries and modules.

## Contents 
1. Configurations (Resources, Training, Optimization, Evaluation, Saving results)
2. Prepare training & dataset
3. Train (Finetune) a NER Model
4. Optimization
5. Evaluation

In [1]:
# import modules
import config
import train
import eval_opt

/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configurations

In this section, we set all of the configurations for the following steps (training, optimization, evaluation). Depending on which parts you want to run and which to leave out, corresponding config parts can be skipped if not needed.

### Resources

We need to state which resources we want to use for NER finetuning, namely the `dataset` and `model`. Therefore, we use the `Resource` class and define the `path` (either to a HF or a local directory) and `source` (`hf` or `local`) of our resources. By applying `set_name()`, the model or dataset name gets set automatically, otherwise the `model.name` can also be updated manually.

In [2]:
"""
model = config.Resource(path="test/test", source="hf")
model.name = "my_model_name"

dataset = config.Resource(path="dir/dataset", source="local")
dataset.set_name()
"""

'\nmodel = config.Resource(path="test/test", source="hf")\nmodel.name = "my_model_name"\n\ndataset = config.Resource(path="dir/dataset", source="local")\ndataset.set_name()\n'

In [2]:
model = config.Resource(path="FacebookAI/xlm-roberta-base", source="hf")
model.set_name()

dataset = config.Resource(path="data/hisgermaner.hf", source="local")
dataset.set_name()

In [3]:
#check our model and dataset configurations again
print(model.info())
print(dataset.info())

xlm-roberta-base will be loaded from FacebookAI/xlm-roberta-base (via hf).
hisgermaner will be loaded from data/hisgermaner.hf (via local).


### Training

You can define a variable for the training parameters based on our default `TrainingParams` config and try to improve results using evaluation/optimization techniques, or start by overwriting the parameters for training manually. Training will only be performed once on the training dataset and evaluated once on the validation dataset. For more sophisticated methods, you can additionally run the Optimization and Evaluation parts to find ideal hyperparameter combinations and perform a more reliable evaluation.

In [4]:
training_params = config.TrainingParams()
training_params.__dict__

{'batch_size': 16,
 'eval_strategy': 'epoch',
 'learning_rate': 2e-05,
 'num_train_epochs': 3,
 'weight_decay': 0.01,
 'warmup_steps': 100}

In [13]:
training_params.batch_size = 32
training_params.num_train_epochs = 6
training_params.__dict__

{'batch_size': 32,
 'eval_strategy': 'epoch',
 'learning_rate': 2e-05,
 'num_train_epochs': 6,
 'weight_decay': 0.01,
 'warmup_steps': 100}

### Optimization

For optimization (hyperparameter search) we use the HF Trainer API (https://huggingface.co/docs/transformers/hpo_train#hyperparameter-search-using-trainer-api) and optuna (https://optuna.readthedocs.io/en/stable/index.html, `pip install optuna`). We can access all of the search parameters, as they are stored in a dict structure, and also adapt the number of trials `n_trials` to run. 

In [6]:
optimize_params = config.OptimizeParams()

optimize_params.hp_space["learning_rate"]["param_type"]
optimize_params.__dict__

{'hp_space': {'learning_rate': {'param_type': 'float',
   'borders': [1e-06, 0.0001]},
  'warmup_steps': {'param_type': 'categorical', 'borders': [20, 40, 160]}},
 'n_trials': 20}

In [7]:
#to remove an hyperparameter from the hp_space dict, use pop() or similar
#optimize_params.hp_space.pop("warmup_steps")
optimize_params.__dict__

{'hp_space': {'learning_rate': {'param_type': 'float',
   'borders': [1e-06, 0.0001]},
  'warmup_steps': {'param_type': 'categorical', 'borders': [20, 40, 160]}},
 'n_trials': 20}

### Evaluation and optimization

* Evaluation: Stratified k-fold Cross-Validation - one of ["None", k] (https://huggingface.co/docs/datasets/v1.11.0/splits.html#examples; https://discuss.huggingface.co/t/k-fold-cross-validation/5765/5, https://github.com/huggingface/datasets/discussions/5940)
* Optimization:
  * https://huggingface.co/docs/transformers/hpo_train
  * https://medium.com/carbon-consulting/transformer-models-hyperparameter-optimization-with-the-optuna-299e185044a8
  * Grid Search (GridSearchCV?), Line Search, hyperopt
* Analyze grid search results using Heatmap or similar

### Saving results
* save configuration
* create/save plots

## 2. Prepare training & dataset

In [8]:
train.set_torch_device()

device(type='cuda')

In [37]:
ner_dataset = train.load_ner_dataset(dataset.path, dataset.source)
ner_dataset["train"][2]

{'id': 2,
 'tokens': ['Nun',
  'legt',
  'er',
  'die',
  'Feder',
  'bei',
  'Seite',
  ',',
  'verschränkt',
  '# schränkt',
  'die',
  'Arme',
  'und',
  'versinkt',
  'in',
  'dumpfes',
  'Hinbrüten',
  '.'],
 'ner_tags': [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6]}

In [10]:
tokenizer = train.get_tokenizer(model.path)

In [11]:
tokenized_dataset = train.prepare_dataset(ner_dataset, tokenizer)
label_list = train.get_label_list(ner_dataset)

In [12]:
label_list

['B-ORG', 'I-ORG', 'I-PER', 'I-LOC', 'B-LOC', 'B-PER', 'O']

## 3. Train (Finetune) a NER Model

To finetune our model for NER, we first need to define our `ner` task (see train.py). 

In [14]:
trained_ner_model, model_out_path = train.train_model(model, dataset, label_list, training_params, tokenized_dataset, tokenizer)

Some weights of XLMRobertaForTokenClassification were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.382481,0.143843,0.125755,0.134192,0.878194
2,No log,0.259025,0.359706,0.393360,0.375781,0.916978
3,No log,0.207591,0.496844,0.554326,0.524013,0.943036
4,No log,0.183222,0.564030,0.602616,0.582685,0.949803
5,No log,0.189389,0.627010,0.588531,0.607161,0.953288
6,No log,0.184201,0.599218,0.616700,0.607833,0.952328


## 4. Optimization

In [ ]:
optimize_params.n_trials = 10

In [ ]:
eval_opt.optimize(optimize_params, training_params, model.path, label_list, tokenizer, tokenized_dataset)

## 5. Evaluation

In [15]:
class_report, errors = eval_opt.compute_metrics_per_tag(trained_ner_model, tokenized_dataset, label_list) 

/data-ssd/sophie.schneider/pyenv/versions/hf-transformers-env/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


example from test dataset: ['Von', 'M', '.', 'G', '.', 'Saphir', '.']
true classification in test dataset: ['O', 'B-PER', 'I-PER', 'I-PER', 'I-PER', 'I-PER', 'I-PER', 'I-PER', 'I-PER', 'O', 'O']
predicted classification based on finetuned model: ['B-PER', 'B-PER', 'I-PER', 'I-PER', 'I-PER', 'I-PER', 'O', 'O', 'O', 'O', 'O']
              precision    recall  f1-score   support

         LOC       0.65      0.82      0.72       444
         ORG       0.00      0.00      0.00        25
         PER       0.65      0.64      0.65       406

   micro avg       0.65      0.71      0.68       875
   macro avg       0.43      0.49      0.46       875
weighted avg       0.63      0.71      0.67       875



In [16]:
eval_opt.save_class_report(class_report, "md", model_out_path)

In [17]:
config.save_train_config(model_out_path, training_params)

In [36]:
#inference example
from transformers import pipeline, AutoModelForTokenClassification

text = "Novelle	von Joh. L. Buchta."

id2label = {0:label_list[0], 1:label_list[1], 2:label_list[2], 3:label_list[3], 4:label_list[4], 5:label_list[5], 6:label_list[6]}
finetuned_model = AutoModelForTokenClassification.from_pretrained("xlm-roberta-base_hisgermaner_2025-04-02_14-45/checkpoint-276", num_labels=7, id2label=id2label)
classifier = pipeline("ner", model=finetuned_model, tokenizer=tokenizer)
classifier(text)

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


[{'entity': 'B-PER',
  'score': np.float32(0.8944226),
  'index': 4,
  'word': '▁Joh',
  'start': 12,
  'end': 15},
 {'entity': 'I-PER',
  'score': np.float32(0.7188401),
  'index': 5,
  'word': '.',
  'start': 15,
  'end': 16},
 {'entity': 'I-PER',
  'score': np.float32(0.95051396),
  'index': 6,
  'word': '▁L',
  'start': 17,
  'end': 18},
 {'entity': 'I-PER',
  'score': np.float32(0.9569103),
  'index': 7,
  'word': '.',
  'start': 18,
  'end': 19},
 {'entity': 'I-PER',
  'score': np.float32(0.9494691),
  'index': 8,
  'word': '▁Buch',
  'start': 20,
  'end': 24},
 {'entity': 'I-PER',
  'score': np.float32(0.95551276),
  'index': 9,
  'word': 'ta',
  'start': 24,
  'end': 26}]